# Práctica guiada — Inferencia y contratos

Este notebook es para el alumnado. Completa las celdas marcadas con `TODO` y responde las preguntas antes de mirar la solución docente.

**Objetivo:** pasar de una fila CSV a una especificación clara para el módulo de inferencia de la clase 2.

## Cómo trabajar

1. Copia esta carpeta como tu espacio de trabajo.
2. Ejecuta `uv sync` desde `problem/starter/` cuando empiece la clase 2.
3. En esta práctica no implementes todavía el CLI completo.
4. Trabaja en parejas y cambia quién escribe cada diez minutos.
5. No abras `solutions/` hasta el debrief.

In [ ]:
from dataclasses import dataclass
import csv
from pathlib import Path

from pydantic import BaseModel, ConfigDict, Field, ValidationError

DATA_PATH = Path("../../../../../../assets/03-wine-quality/inference_samples.csv")
print(DATA_PATH)
print(DATA_PATH.exists())

## 1. Leer los datos nuevos

### Práctica A — Inspección del CSV

Completa la celda siguiente para:

- Leer el CSV con `csv.DictReader`.
- Imprimir el número de filas.
- Imprimir las columnas.
- Guardar la primera fila en una variable llamada `row`.

In [ ]:
# TODO: implementa la lectura del CSV.
# Pista: usa DATA_PATH.open(newline="", encoding="utf-8")
with DATA_PATH.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    columns = reader.fieldnames
    rows = list(reader)

row = rows[0]
print("Filas:", len(rows))
print("Columnas:", columns)
print("Primera fila:", row)


### Preguntas

1. ¿Qué campo identifica la muestra pero no es una característica del modelo? Sample id
2. ¿Cuántas características numéricas hay? 11
3. ¿Qué columna extra añadirías para probar el contrato? unexpected field
4. ¿Qué valor de `ph` sería claramente inválido? un valor negativo o fuera de rango

## 2. Construir el contrato con Pydantic

El contrato del modelo contiene estas características:

| Campo | Rango orientativo |
|---|---|
| `fixed_acidity` | 0–20 |
| `volatile_acidity` | 0–2 |
| `citric_acid` | 0–2 |
| `residual_sugar` | 0–20 |
| `chlorides` | 0–1 |
| `free_sulfur_dioxide` | 0–100 |
| `total_sulfur_dioxide` | 0–300 |
| `density` | 0.98–1.01 |
| `ph` | 2.5–4.5 |
| `sulphates` | 0–3 |
| `alcohol` | 5–20 |

In [ ]:
class WineQualityRequestPydantic(BaseModel):
    model_config = ConfigDict(extra="forbid")

    fixed_acidity: float = Field(ge=0, le=20, allow_inf_nan=False)
    volatile_acidity: float = Field(ge=0, le=2, allow_inf_nan=False)
    citric_acid: float = Field(ge=0, le=2, allow_inf_nan=False)
    residual_sugar: float = Field(ge=0, le=20, allow_inf_nan=False)
    chlorides: float = Field(ge=0, le=1, allow_inf_nan=False)
    free_sulfur_dioxide: float = Field(ge=0, le=100, allow_inf_nan=False)
    total_sulfur_dioxide: float = Field(ge=0, le=300, allow_inf_nan=False)
    density: float = Field(ge=0.98, le=1.01, allow_inf_nan=False)
    ph: float = Field(ge=2.5, le=4.5, allow_inf_nan=False)
    sulphates: float = Field(ge=0, le=3, allow_inf_nan=False)
    alcohol: float = Field(ge=5, le=20, allow_inf_nan=False)


model_row = {k: v for k, v in row.items() if k != "sample_id"}
request = WineQualityRequestPydantic.model_validate(model_row)
print(request)
    # TODO: prohíbe campos extra con model_config.
    # TODO: declara las 11 características con Field(ge=..., le=...).


# TODO: separa sample_id de la fila y valida solo las características del modelo.
# model_row = ...
# request = WineQualityRequestPydantic.model_validate(model_row)

### Práctica B — Validación

Completa una prueba para cada caso:

- Una fila válida.
- Una columna `unexpected_field`.
- Un `ph` fuera de rango.

Antes de ejecutar, escribe qué error esperas.

In [ ]:
# TODO: completa los casos y captura ValidationError.
# try:
#     ...
# except ValidationError as error:
#     print(error)

casos = {
    "fila valida": dict(model_row),
    "columna extra": {**model_row, "unexpected_field": 1},
    "ph fuera de rango": {**model_row, "ph": 99.0},
}

for nombre, datos in casos.items():
    try:
        WineQualityRequestPydantic.model_validate(datos)
        print(nombre, "-> OK")
    except ValidationError as error:
        print(nombre, "-> ERROR")
        print(error)

## 3. Comparar con `dataclass`

Construye la misma estructura con `dataclass`. No añadas todavía validación manual: observa qué garantiza Python y qué no garantiza.

In [ ]:
@dataclass(frozen=True, slots=True)
class WineQualityRequestDataclass:
    fixed_acidity: float
    volatile_acidity: float
    citric_acid: float
    residual_sugar: float
    chlorides: float
    free_sulfur_dioxide: float
    total_sulfur_dioxide: float
    density: float
    ph: float
    sulphates: float
    alcohol: float


datos = {k: float(v) for k, v in model_row.items()}
ok = WineQualityRequestDataclass(**datos)
print(ok)

malo = WineQualityRequestDataclass(**{**datos, "ph": 99.0})
print(malo)  # no da error: la dataclass no valida rangos
    # TODO: declara las mismas 11 características.
    

# TODO: crea una instancia válida y prueba qué ocurre con ph=99.0.

### Decisión de diseño

Completa la frase:

> Usaría Pydantic en csv porque valida y rechaza. Usaría `dataclass` en estructuras ya validadas porque mas ligero y sencillo.

## 4. Definir el orden del vector

El modelo no recibe un diccionario. Recibe una lista numérica con un orden estable.

In [ ]:
# TODO: rellena el orden exacto a partir del CSV y del contrato.
FEATURE_NAMES = (
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
    # "fixed_acidity",
    # ...
)

vector = [getattr(request, name) for name in FEATURE_NAMES]
print(len(vector), vector)

# TODO: construye vector usando getattr(request, name).
# vector = [...]

### Práctica C — Error de orden

Intercambia mentalmente `density` y `alcohol`. Responde:

- ¿El vector sigue teniendo 11 valores?
- ¿El clasificador detectaría necesariamente el error?
- ¿Cómo evitarías que cada script definiera un orden distinto?

## 5. Contrato de salida

Diseña la estructura que devolverá el módulo. Debe permitir responder:

- ¿Qué muestra se ha procesado?
- ¿Qué categoría se ha predicho?
- ¿Con qué confianza?
- ¿Qué versión de modelo y preprocesado se utilizaron?

In [ ]:
class WineQualityPredictionPydantic(BaseModel):
    model_config = ConfigDict(extra="forbid")

    sample_id: str = Field(min_length=1)
    category: str = Field(min_length=1)
    confidence: float = Field(ge=0, le=1, allow_inf_nan=False)
    model_version: str = Field(min_length=1)
    preprocess_version: str = Field(min_length=1)


ejemplo = WineQualityPredictionPydantic(
    sample_id="red-001",
    category="ejemplo",  # valor ilustrativo
    confidence=0.5,      # valor ilustrativo
    model_version="v1",
    preprocess_version="v1",
)
print(ejemplo.model_dump())    # TODO: declara los campos de salida y sus restricciones.


# TODO: escribe un ejemplo de salida para red-001.

## 6. Demo del docente: observar la inferencia

El docente ejecutará el notebook resuelto y el comando de referencia. Mientras tanto, anota el recorrido:

```text
CSV → contrato → preprocesado → modelo cargado → predicción
```

No copies la implementación. Tu código de producción empieza en la clase 2 con el starter.

## Entrega

Entrega una captura o exportación con:

- La tabla de contrato.
- Las respuestas de las prácticas A, B y C.
- El orden de las 11 características.
- Un ejemplo de entrada inválida.
- Un contrato de salida.
- Tu plan de implementación para `contracts.py`, `preprocess.py`, `inference.py` y `predict_file.py`.